In [64]:
# Cell 1: Imports and normalization setup

import os
import json
from collections import defaultdict, Counter
from tqdm import tqdm

target_genres = [
    'rock', 'pop', 'jazz', 'electronic', 'blues', 'metal', 'hiphop', 'country', 
    'folk', 'soul', 'punk', 'classical', 'reggae', 'rnb', 'indie', 'funk', 
    'dance', 'latin', 'ambient', 'experimental', 'world'
]

explicit_map = {
    'pink': 'pop',
    'p!nk': 'pop',
    'beatles': 'rock',
    'the beatles': 'rock',
    'stones': 'rock',
    'the stones': 'rock',
    'alternative': 'rock',
    'emo': 'punk',
    'shoegaze': 'rock',
    'acoustic': 'folk',
    'instrumental': 'other',
    'soundtrack': 'other',
    'british': 'pop',
    '80s': 'pop',
    "80's": 'pop',
    '90s': 'rock',
    '90': 'rock',
    '70s': 'pop',
    '70': 'pop',
    # Add more mappings as you need
}

def normalize_tag(tag):
    t = tag.strip().lower()
    if t in explicit_map:
        return explicit_map[t]
    if t in target_genres:
        return t
    for genre in target_genres:
        if genre in t:
            return genre
    if 'vocal' in t or 'female' in t or 'male' in t:
        return 'pop'
    if 'singer' in t or 'songwriter' in t:
        return 'folk'
    if 'electronica' in t or 'electro' in t:
        return 'electronic'
    if 'blues' in t:
        return 'blues'
    if 'funk' in t:
        return 'funk'
    if 'ambient' in t or 'chill' in t:
        return 'ambient'
    if 'latin' in t or 'bossa' in t or 'samba' in t or 'salsa' in t or 'reggaeton' in t:
        return 'latin'
    if 'jazz' in t:
        return 'jazz'
    if 'punk' in t:
        return 'punk'
    if 'metal' in t:
        return 'metal'
    if 'rock' in t:
        return 'rock'
    if 'pop' in t:
        return 'pop'
    if 'country' in t:
        return 'country'
    if 'folk' in t:
        return 'folk'
    if 'indie' in t:
        return 'indie'
    if 'experimental' in t:
        return 'experimental'
    if 'dance' in t:
        return 'dance'
    if 'reggae' in t:
        return 'reggae'
    if 'soul' in t:
        return 'soul'
    if 'rnb' in t or 'r&b' in t:
        return 'rnb'
    if 'hiphop' in t or 'hip-hop' in t or 'rap' in t:
        return 'hiphop'
    return 'other'


In [65]:
# Cell 2: Trigram extraction function

def extract_functional_trigrams(chords):
    """
    Extract trigrams where all three functionals are distinct (case-insensitive).
    """
    funcs = [
        c["functional_harmony"]["functional"]
        for c in chords
        if "functional_harmony" in c and "functional" in c["functional_harmony"]
    ]
    trigrams = []
    for i in range(len(funcs) - 2):
        tri = tuple(funcs[i:i+3])
        lower = [x.lower() for x in tri]
        if len(set(lower)) == 3:
            trigrams.append(tri)
    return trigrams



In [66]:
# Cell 3: Style JSON loader

def load_style_json(style_path):
    with open(style_path, "r") as f:
        data = json.load(f)
    # Human tags (to normalize)
    original = data.get("original", [])
    # Dortmund genres (use as-is, floats)
    genre_dortmund = {k: float(v) for k, v in data.get("genre_dortmund", {}).items()}
    return original, genre_dortmund


In [67]:
# Cell 4: Main dataset builder

dataset_root = "/workspace/dataset_corrected/lastfm"
output_json = "/workspace/dataset/trigrams/trigram_dataset_lastfm.json"

trigram_data = defaultdict(lambda: {
    "function": None,
    "count": 0,
    "styles": Counter(),
    "genres": Counter(),
    "song_ids": Counter()   # song_id: count in this song
})

all_song_ids = [
    d for d in os.listdir(dataset_root)
    if os.path.isdir(os.path.join(dataset_root, d))
]

for song_id in tqdm(all_song_ids, desc="Processing songs"):
    # Chord path
    chords_path = os.path.join(dataset_root, song_id, f"{song_id}_normalized.json")
    style_path  = os.path.join(dataset_root, song_id, f"{song_id}_style.json")
    # Skip if files are missing
    if not (os.path.exists(chords_path) and os.path.exists(style_path)):
        continue

    # Load chord and style data
    with open(chords_path, "r") as f:
        chords_data = json.load(f)
    chords = chords_data.get("chords", [])
    trigrams = extract_functional_trigrams(chords)

    original, genre_dortmund = load_style_json(style_path)
    normalized_styles = [normalize_tag(tag) for tag in original]

    for tri in trigrams:
        key = tuple(tri)
        trigram_data[key]["function"] = list(key)
        trigram_data[key]["count"] += 1
        trigram_data[key]["song_ids"][song_id] += 1

        for genre in normalized_styles:
            if genre != "other":  # skip "other" if not meaningful
                trigram_data[key]["styles"][genre] += 1
        for dortmund_genre, value in genre_dortmund.items():
            trigram_data[key]["genres"][dortmund_genre] += value


Processing songs: 100%|██████████| 19913/19913 [01:02<00:00, 317.57it/s]


In [68]:
# Cell 5: Export to JSON

export_trigrams = []
for trigram, entry in trigram_data.items():
    export_trigrams.append({
        "function": entry["function"],
        "count": entry["count"],
        "styles": sorted(entry["styles"].items(), key=lambda x: -x[1]),
        "genres": sorted(entry["genres"].items(), key=lambda x: -x[1]),
        "song_ids": sorted(entry["song_ids"].items(), key=lambda x: -x[1]),
    })

# Sort by count descending
export_trigrams.sort(key=lambda x: -x["count"])

# Write output
os.makedirs(os.path.dirname(output_json), exist_ok=True)
with open(output_json, "w") as f:
    json.dump(export_trigrams, f, indent=2)

print(f"Exported {len(export_trigrams)} trigrams to {output_json}")


Exported 45840 trigrams to /workspace/dataset/trigrams/trigram_dataset_lastfm.json


In [69]:
# Cell 6: (Optional) Quick check on "other" tags

all_other = []
for entry in export_trigrams:
    for style, count in entry["styles"]:
        if style == "other":
            all_other.append((entry["function"], count))
if all_other:
    print("Examples of trigrams with 'other' style (fix explicit_map to reduce):")
    for trigram, count in all_other[:20]:
        print(trigram, count)
else:
    print("No 'other' tags found!")


No 'other' tags found!


In [73]:
from trigram_search_lastfm import query_lastfm_trigram_usage

query = ["ii", "V", "I"]
result = query_lastfm_trigram_usage(query)
print(json.dumps(result, indent=2))

{
  "count": 572,
  "styles": [
    [
      "rock",
      923
    ],
    [
      "pop",
      743
    ],
    [
      "metal",
      272
    ],
    [
      "folk",
      199
    ],
    [
      "indie",
      158
    ],
    [
      "ambient",
      140
    ],
    [
      "electronic",
      106
    ],
    [
      "jazz",
      78
    ],
    [
      "country",
      76
    ],
    [
      "punk",
      73
    ]
  ],
  "genres": [
    [
      "rock",
      170.29234488310001
    ],
    [
      "folkcountry",
      73.10825485958995
    ],
    [
      "jazz",
      72.82324892740006
    ],
    [
      "pop",
      62.13661981761402
    ],
    [
      "electronic",
      60.239820133209996
    ],
    [
      "alternative",
      46.94672905819997
    ],
    [
      "raphiphop",
      30.87705235279599
    ],
    [
      "blues",
      28.275530381829967
    ],
    [
      "funksoulrnb",
      24.027759504240002
    ]
  ],
  "matched": "exact",
  "function": [
    "ii",
    "V",
    "I"
  ],
 